# Assignment 5 — Add-K Smoothing (K = 0.3)



> Apply **Add-K smoothing with K = 0.3** to all four models from the previous assignment, and evaluate them on the **test** and **development/validation** sets.

$$P_{\text{Add-K}}(w_n \mid w_1,\dots,w_{n-1}) = \frac{C(w_1,\dots,w_n)+K}{C(w_1,\dots,w_{n-1})+K \cdot V}$$

with **K = 0.3**. For the unigram model there is no context, so the denominator is $N + K V$, where $N$ is the number of training tokens.


## 0. Setup and imports

In [ ]:

import os
import math
import time
import random
from collections import Counter, defaultdict

import pandas as pd

K = 0.3  # Add-K smoothing constant from the assignment

print(f"Setup complete. K = {K}")

Setup complete. K = 0.3


## 1. Data, split and the four n-gram models

Same pipeline as Assignment 4 so the models being smoothed are the ones from the previous lab: **1,000,000** tokenized Telugu sentences, **1,000** development, **1,000** test, remainder training.


In [2]:
data_path = os.path.join(
    "..", "Assignment1", "output", "indiccorp", "indiccorp_te_tokenized.parquet"
)

df = pd.read_parquet(data_path)
df_subset = df.head(1_000_000)

all_sents = []
for text in df_subset["tokenized_sentence"]:
    if isinstance(text, str):
        tokens = text.split()
        if tokens:
            all_sents.append(tokens)

print(f"Total sentences loaded: {len(all_sents)}")
print("Example sentence:", all_sents[0] if all_sents else None)

Total sentences loaded: 1000000
Example sentence: ['అమెరికా', 'అధ్యక్షుడు', 'డొనాల్డ్', 'ట్రంప్', 'కు', 'రాష్ట్రపతి', 'భవన్', 'వద్ద', 'ఘనస్వాగతం', 'లభించింది', '.']


### 1.1 Train / development / test split

In [3]:
random.seed(42)
random.shuffle(all_sents)

test_set = all_sents[:1000]
dev_set = all_sents[1000:2000]
train_set = all_sents[2000:]

print(f"Train sentences: {len(train_set)}")
print(f"Dev sentences:   {len(dev_set)}")
print(f"Test sentences:  {len(test_set)}")
assert len(dev_set) == 1000 and len(test_set) == 1000
assert len(train_set) == len(all_sents) - 2000

Train sentences: 998000
Dev sentences:   1000
Test sentences:  1000


### 1.2 Vocabulary and OOV handling

Vocabulary is built **only from training**. Unseen development/test words map to `<UNK>`. `<s>` and `</s>` pad sentence edges for higher-order models.


In [4]:
vocab = set()
for s in train_set:
    vocab.update(s)

vocab.add("<UNK>")
vocab.add("<s>")
vocab.add("</s>")
V = len(vocab)

print(f"Vocabulary size (V): {V:,}")


def replace_unk(sentence):
    """Map any word not seen in training to <UNK>."""
    return [w if w in vocab else "<UNK>" for w in sentence]


def pad(sentence, n):
    """Add (n-1) start markers and one end marker."""
    return ["<s>"] * (n - 1) + sentence + ["</s>"]

Vocabulary size (V): 591,166


### 1.3 Count n-grams for unigram, bigram, trigram, and quadrigram

In [5]:
def build_ngram_counts(sentences, n):
    ngram_counts = Counter()
    context_counts = Counter()
    for s in sentences:
        padded = pad(replace_unk(s), n)
        for i in range(n - 1, len(padded)):
            ngram = tuple(padded[i - n + 1 : i + 1])
            context = ngram[:-1]
            ngram_counts[ngram] += 1
            context_counts[context] += 1
    return ngram_counts, context_counts


models = {}  # n -> (ngram_counts, context_counts, context_to_words)
for n in [1, 2, 3, 4]:
    t0 = time.time()
    ngc, cc = build_ngram_counts(train_set, n)
    context_to_words = defaultdict(set)
    for ngram in ngc:
        context_to_words[ngram[:-1]].add(ngram[-1])
    models[n] = (ngc, cc, context_to_words)
    name = {1: "Unigram", 2: "Bigram", 3: "Trigram", 4: "Quadrigram"}[n]
    print(
        f"{name:10s} (n={n}): {len(ngc):>11,} unique n-grams, "
        f"{len(cc):>11,} unique contexts  (built in {time.time() - t0:.1f}s)"
    )

UNIGRAM_TOTAL_TOKENS = sum(models[1][0].values())
print(f"\nTotal training tokens (unigram N): {UNIGRAM_TOTAL_TOKENS:,}")

Unigram    (n=1):     591,164 unique n-grams,           1 unique contexts  (built in 10.8s)


Bigram     (n=2):   4,777,970 unique n-grams,     591,164 unique contexts  (built in 21.9s)


Trigram    (n=3):   8,394,190 unique n-grams,   4,759,819 unique contexts  (built in 33.2s)


Quadrigram (n=4):  10,141,000 unique n-grams,   8,251,120 unique contexts  (built in 35.5s)

Total training tokens (unigram N): 13,330,633


## 2. Apply Add-K smoothing (K = 0.3)

$$P_{\text{Add-K}}(w_n \mid w_1,\dots,w_{n-1}) = \frac{C(w_1,\dots,w_n)+K}{C(w_1,\dots,w_{n-1})+K \cdot V}, \quad K=0.3$$

Compared with Laplace (Assignment 4), Add-K is the same family of estimators with a smaller additive constant. $K < 1$ puts less probability mass on unseen n-grams, which usually helps higher-order models on a large vocabulary.


In [6]:
def add_k_prob(ngram, n, k=K):
    """Add-K smoothed P(last word | context)."""
    ngc, cc, _ = models[n]
    if n == 1:
        return (ngc[ngram] + k) / (UNIGRAM_TOTAL_TOKENS + k * V)
    context = ngram[:-1]
    return (ngc[ngram] + k) / (cc[context] + k * V)


most_common_word = models[1][0].most_common(1)[0][0][0]
print(
    f'P("{most_common_word}") under Add-K unigram (K={K}) = '
    f"{add_k_prob((most_common_word,), 1):.8f}"
)

P(".") under Add-K unigram (K=0.3) = 0.08328833


### 2.1 Perplexity

$$PP = \exp\left(-\frac{1}{N}\sum_{i}\log P(w_i \mid \text{context})\right)$$


In [7]:
def sentence_log_prob(sentence, n):
    padded = pad(replace_unk(sentence), n)
    log_prob, count = 0.0, 0
    for i in range(n - 1, len(padded)):
        ngram = tuple(padded[i - n + 1 : i + 1])
        log_prob += math.log(add_k_prob(ngram, n))
        count += 1
    return log_prob, count


def perplexity(sentences, n):
    total_log_prob, total_tokens = 0.0, 0
    for s in sentences:
        lp, c = sentence_log_prob(s, n)
        total_log_prob += lp
        total_tokens += c
    return math.exp(-total_log_prob / total_tokens)

### 2.2 Evaluate all four models on development and test sets

In [8]:
results = []
for n in [1, 2, 3, 4]:
    name = {1: "Unigram", 2: "Bigram", 3: "Trigram", 4: "Quadrigram"}[n]
    t0 = time.time()
    pp_dev = perplexity(dev_set, n)
    pp_test = perplexity(test_set, n)
    results.append((name, pp_dev, pp_test))
    print(f"{name} evaluated in {time.time() - t0:.1f}s")

print()
print(f"Add-K smoothing with K = {K}")
print(f"{'Model':<12}{'Dev Perplexity':>18}{'Test Perplexity':>18}")
print("-" * 48)
for name, pp_dev, pp_test in results:
    print(f"{name:<12}{pp_dev:>18.2f}{pp_test:>18.2f}")

Unigram evaluated in 0.0s
Bigram evaluated in 0.0s
Trigram evaluated in 0.0s
Quadrigram evaluated in 0.0s

Add-K smoothing with K = 0.3
Model           Dev Perplexity   Test Perplexity
------------------------------------------------
Unigram                4272.55           4341.33
Bigram                10388.86          10362.17
Trigram               76427.28          77076.44
Quadrigram           173770.04         177312.72


### 2.3 Discussion

Add-K with **K = 0.3** is milder than Laplace (K = 1):

- The extra mass on unseen events is $K V$ instead of $V$, so higher-order models waste less probability on n-grams that never occur.
- Unigrams still have dense counts, so their perplexity usually changes less than bigram/trigram/quadrigram.
- If K is too small, unseen events can be under-smoothed; 0.3 is a common compromise between Laplace and a tiny K.

Compare these perplexities with Assignment 4 (Laplace). Lower numbers here, especially for $n \ge 2$, mean Add-K is a better fit for this Telugu vocabulary.


### 2.4 Example: next-word predictions and sentence probability

In [9]:
def top_next_words(context_tuple, n, k_top=5):
    """k most probable next words given a fixed (n-1)-word context."""
    _, _, context_to_words = models[n]
    seen = set(context_to_words.get(context_tuple, set()))
    for (w,), _ in models[1][0].most_common(20):
        seen.add(w)
    seen.update(["<UNK>", "</s>"])

    candidates = [(w, add_k_prob(context_tuple + (w,), n)) for w in seen]
    candidates.sort(key=lambda x: x[1], reverse=True)
    return candidates[:k_top]


sample_context = None
for ngram, _ in models[2][0].most_common(50):
    if ngram[0] not in {"<s>", "</s>", "<UNK>"}:
        sample_context = (ngram[0],)
        break

print(f"Top 5 words after {sample_context} (bigram, Add-K):")
for w, p in top_next_words(sample_context, 2):
    print(f"  {w:<20} P = {p:.8f}")

tri_ctx = None
for ngram, _ in models[3][0].most_common(80):
    if ngram[0] not in {"<s>", "</s>"} and ngram[1] not in {"<s>", "</s>"}:
        tri_ctx = ngram[:2]
        break

print(f"\nTop 5 words after {tri_ctx} (trigram, Add-K):")
for w, p in top_next_words(tri_ctx, 3):
    print(f"  {w:<20} P = {p:.8f}")

sample = test_set[0]
print(f"\nSample test sentence: {' '.join(sample)}")
for n in [1, 2, 3, 4]:
    lp, c = sentence_log_prob(sample, n)
    name = {1: "Unigram", 2: "Bigram", 3: "Trigram", 4: "Quadrigram"}[n]
    print(f"  {name:<10} log P(sentence) = {lp:.2f}   ({c} tokens)")

Top 5 words after ('.',) (bigram, Add-K):
  </s>                 P = 0.69283138
  .                    P = 0.10633874
  మీ                   P = 0.00934447
  కి                   P = 0.00295246
  ?                    P = 0.00224300



Top 5 words after ('.', '.') (trigram, Add-K):
  </s>                 P = 0.30647962
  .                    P = 0.08095184
  ?                    P = 0.00809351
  !                    P = 0.00621601
  ’                    P = 0.00164732

Sample test sentence: మాతా శిశు సంరక్షణ కేంద్రం గ్రామం నుండి 10 కి . మీ .
  Unigram    log P(sentence) = -76.32   (12 tokens)
  Bigram     log P(sentence) = -50.66   (12 tokens)
  Trigram    log P(sentence) = -59.72   (12 tokens)
  Quadrigram log P(sentence) = -68.43   (12 tokens)
